# 31-Day Inventory Online Learning Challenge

This notebook compares five online learning strategies for choosing a daily order-up-to level: Epsilon-Greedy, UCB, Gradient Bandit, EWF, and FSF. Each morning, a strategy uses only the day number and inventory carried into the day. Sales and profit are revealed only after the decision. True demand remains private to the environment and is used separately for evaluation.

## Imports, configuration, and challenge data

The demand sequence is embedded to keep the notebook self-contained. The economic settings match the project implementation: zero lead time, lost sales, and a holding charge on ending inventory.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PRICE = 10.0
UNIT_COST = 4.0
HOLDING_COST = 1.0
INITIAL_INVENTORY = 0
N_DAYS = 31
ACTION_SET = list(range(0, 50_001, 1_000))
RANDOM_SEED = 42

demand_data = pd.DataFrame({
    "date": range(1, N_DAYS + 1),
    "demand": [
        47_995, 20_714, 22_284, 22_255, 24_400, 23_265, 29_452, 36_541,
        23_598, 13_411, 13_577, 17_285, 18_017, 16_912, 16_199, 16_039,
        13_916, 12_611, 15_962, 15_021, 18_129, 18_884, 23_336, 18_584,
        22_076, 22_755, 25_055, 29_569, 24_829, 23_619, 22_865,
    ],
})
assert len(demand_data) == N_DAYS

## Shared strategy interface and inventory environment

For target level $q_t$, the order quantity is $\max(0, q_t-I_t)$. Available inventory is sold up to demand, and unsold units carry forward. Daily profit is revenue minus ordering and holding costs. The policy-facing state and observation deliberately exclude true demand.

In [ ]:
class BaseStrategy:
    """Common interface and action helpers for all five strategies."""

    name = "base"

    def __init__(self, action_set, rng=None):
        self.actions = list(action_set)
        self.n_actions = len(self.actions)
        self.action_index = {action: i for i, action in enumerate(self.actions)}
        self.rng = rng if rng is not None else np.random.default_rng()

    def _resolve_actions(self, available_actions=None):
        actions = list(available_actions) if available_actions is not None else self.actions
        if not actions:
            raise ValueError("available_actions cannot be empty")
        return actions

    def _argmax_random(self, values):
        values = np.asarray(values, dtype=float)
        best = np.flatnonzero(values == values.max())
        return int(self.rng.choice(best))

    def select_action(self, state, available_actions=None):
        raise NotImplementedError

    def update(self, observation):
        raise NotImplementedError


class InventoryEnvironment:
    """Inventory simulator that keeps demand private from each strategy."""

    def __init__(self, demand, price, unit_cost, holding_cost, initial_inventory=0):
        self._demand = [int(value) for value in demand]
        self.price = float(price)
        self.unit_cost = float(unit_cost)
        self.holding_cost = float(holding_cost)
        self.initial_inventory = int(initial_inventory)
        self.n_days = len(self._demand)
        self.reset()

    def reset(self):
        self.day = 1
        self.inventory = self.initial_inventory
        self.done = False
        self._last_record = None
        return self.get_state()

    def get_state(self):
        return {"day": self.day, "inventory_before": self.inventory}

    def step(self, order_up_to):
        if self.done:
            raise RuntimeError("The simulation has already finished")
        order_up_to = int(order_up_to)
        inventory_before = self.inventory
        order_quantity = max(0, order_up_to - inventory_before)
        available = inventory_before + order_quantity
        demand_today = self._demand[self.day - 1]
        sales = min(demand_today, available)
        inventory_after = available - sales
        profit = (
            self.price * sales
            - self.unit_cost * order_quantity
            - self.holding_cost * inventory_after
        )
        observation = {
            "day": self.day,
            "inventory_before": inventory_before,
            "order_up_to": order_up_to,
            "order_quantity": order_quantity,
            "sales": sales,
            "inventory_after": inventory_after,
            "profit": profit,
        }
        self._last_record = {**observation, "demand": demand_today}
        self.inventory = inventory_after
        if self.day == self.n_days:
            self.done = True
        else:
            self.day += 1
        return observation

    def get_last_record(self):
        return dict(self._last_record)

    def profit_upper_bound(self):
        return float(sum((self.price - self.unit_cost) * d for d in self._demand))

## Algorithm 1: Epsilon-Greedy

With probability $\epsilon$, choose a random level; otherwise choose a level with the highest estimated mean profit. Ties are resolved reproducibly at random.

In [ ]:
class EpsilonGreedy(BaseStrategy):
    name = "Epsilon-Greedy"

    def __init__(self, action_set, epsilon=0.0857, optimistic_init=0.0, rng=None):
        super().__init__(action_set, rng=rng)
        self.epsilon = float(epsilon)
        self.counts = np.zeros(self.n_actions, dtype=float)
        self.values = np.full(self.n_actions, float(optimistic_init), dtype=float)

    def select_action(self, state, available_actions=None):
        actions = self._resolve_actions(available_actions)
        indices = [self.action_index[action] for action in actions]
        if self.rng.random() < self.epsilon:
            return actions[int(self.rng.integers(len(actions)))]
        values = np.array([self.values[i] for i in indices])
        return actions[self._argmax_random(values)]

    def update(self, observation):
        index = self.action_index[observation["order_up_to"]]
        reward = float(observation["profit"])
        self.counts[index] += 1
        self.values[index] += (reward - self.values[index]) / self.counts[index]

## Algorithm 2: Upper Confidence Bound (UCB)

UCB adds a confidence bonus to each estimated mean profit. Untried levels receive priority, while the coefficient $c$ controls subsequent exploration.

In [ ]:
class UCB(BaseStrategy):
    name = "UCB"

    def __init__(self, action_set, c=100.0, optimistic_init=0.0, rng=None):
        super().__init__(action_set, rng=rng)
        self.c = float(c)
        self.counts = np.zeros(self.n_actions, dtype=float)
        self.values = np.full(self.n_actions, float(optimistic_init), dtype=float)

    def select_action(self, state, available_actions=None):
        actions = self._resolve_actions(available_actions)
        indices = [self.action_index[action] for action in actions]
        unplayed = [i for i in indices if self.counts[i] == 0]
        if unplayed:
            return actions[int(self.rng.integers(len(unplayed)))]
        time = self.counts.sum() + 1.0
        scores = np.array([
            self.values[i] + self.c * math.sqrt(math.log(time) / self.counts[i])
            for i in indices
        ])
        return actions[self._argmax_random(scores)]

    def update(self, observation):
        index = self.action_index[observation["order_up_to"]]
        reward = float(observation["profit"])
        self.counts[index] += 1
        self.values[index] += (reward - self.values[index]) / self.counts[index]

## Algorithm 3: Gradient Bandit

The gradient strategy learns a preference for every level and samples from a softmax distribution. Profit is scaled for numerical stability, and the running mean reward is used as a baseline.

In [ ]:
class GradientBandit(BaseStrategy):
    name = "Gradient Bandit"

    def __init__(self, action_set, alpha=3.1089553618622863, reward_scale=10_000.0, use_baseline=True, rng=None):
        super().__init__(action_set, rng=rng)
        self.alpha = float(alpha)
        self.reward_scale = float(reward_scale) if reward_scale else 1.0
        self.use_baseline = bool(use_baseline)
        self.preferences = np.zeros(self.n_actions, dtype=float)
        self._baseline_sum = 0.0
        self._baseline_count = 0

    def _softmax(self, indices):
        preferences = np.array([self.preferences[i] for i in indices])
        weights = np.exp(preferences - preferences.max())
        return weights / weights.sum()

    @property
    def baseline(self):
        return self._baseline_sum / self._baseline_count if self._baseline_count else 0.0

    def select_action(self, state, available_actions=None):
        actions = self._resolve_actions(available_actions)
        indices = [self.action_index[action] for action in actions]
        probabilities = self._softmax(indices)
        return actions[int(self.rng.choice(len(actions), p=probabilities))]

    def update(self, observation):
        index = self.action_index[observation["order_up_to"]]
        reward = float(observation["profit"]) / self.reward_scale
        baseline = 0.0
        if self.use_baseline:
            self._baseline_sum += reward
            self._baseline_count += 1
            baseline = self.baseline
        probabilities = self._softmax(range(self.n_actions))
        indicator = np.zeros(self.n_actions)
        indicator[index] = 1.0
        self.preferences += self.alpha * (reward - baseline) * (indicator - probabilities)

## Algorithm 4: Exponentially Weighted Forecaster (EWF)

EWF samples from exponentially weighted action probabilities with uniform exploration. On a non-stockout day, sales reveal demand and all levels receive an exact newsvendor-loss update. On a stockout day, its censored-feedback estimator updates only levels no greater than the selected target.

In [ ]:
class ExponentiallyWeightedForecaster(BaseStrategy):
    name = "EWF"

    def __init__(
        self, action_set, eta=0.00042919342601287783, gamma=0.02,
        share_alpha=0.0, overage=None, underage=None, rng=None,
    ):
        super().__init__(action_set, rng=rng)
        self.overage = float(UNIT_COST + HOLDING_COST if overage is None else overage)
        self.underage = float(PRICE - UNIT_COST if underage is None else underage)
        self.beta = float(max(self.actions)) * max(self.overage, self.underage)
        self.eta = float(eta)
        self.gamma = float(gamma)
        self.share_alpha = float(share_alpha)
        self._log_weights = np.zeros(self.n_actions, dtype=float)
        self._selection_probabilities = np.full(self.n_actions, 1.0 / self.n_actions)

    def _softmax(self):
        centered = self._log_weights - self._log_weights.max()
        weights = np.exp(centered)
        return weights / weights.sum()

    def _exact_cost(self, demand):
        levels = np.asarray(self.actions, dtype=float)
        difference = levels - demand
        return (
            self.overage * np.maximum(difference, 0.0)
            + self.underage * np.maximum(-difference, 0.0)
        )

    def _recover_demand(self, observation):
        sales = float(observation["sales"])
        available = float(observation["inventory_before"] + observation["order_quantity"])
        return sales if sales < available - 1e-9 else None

    def _censored_cost(self, observation):
        selected_level = float(observation["order_up_to"])
        levels = np.asarray(self.actions, dtype=float)
        surrogate = -self.underage * levels
        estimates = np.zeros(self.n_actions, dtype=float)
        valid = levels <= selected_level
        probability_at_least = np.cumsum(self._selection_probabilities[::-1])[::-1]
        estimates[valid] = (surrogate[valid] + self.beta) / probability_at_least[valid]
        return estimates

    def select_action(self, state, available_actions=None):
        actions = self._resolve_actions(available_actions)
        indices = [self.action_index[action] for action in actions]
        probabilities = (1.0 - self.gamma) * self._softmax() + self.gamma / self.n_actions
        if len(indices) < self.n_actions:
            restricted = probabilities[indices]
            restricted /= restricted.sum()
            probabilities = np.zeros(self.n_actions)
            probabilities[indices] = restricted
        self._selection_probabilities = probabilities.copy()
        return actions[int(self.rng.choice(len(indices), p=probabilities[indices]))]

    def update(self, observation):
        recovered_demand = self._recover_demand(observation)
        costs = (
            self._exact_cost(recovered_demand)
            if recovered_demand is not None
            else self._censored_cost(observation)
        )
        self._log_weights -= self.eta * costs
        if self.share_alpha > 0:
            probabilities = self._softmax()
            weighted = probabilities * np.exp(-self.eta * costs)
            alpha = self.share_alpha
            probabilities = (weighted + alpha / self.n_actions) / (weighted.sum() + alpha)
            probabilities = np.clip(probabilities, 0.0, 1.0)
            self._log_weights = np.log(probabilities / probabilities.sum())

## Algorithm 5: Fixed-Share Forecaster (FSF)

FSF uses the same censored-feedback updates as EWF and adds a fixed probability share after each update. This allows previously weak levels to regain probability when the demand pattern changes.

In [ ]:
class FixedShareForecaster(ExponentiallyWeightedForecaster):
    name = "FSF"

    def __init__(self, action_set, share_alpha=1.0 / N_DAYS, **kwargs):
        super().__init__(action_set, share_alpha=share_alpha, **kwargs)

## Common simulation and evaluation

The event order below is the central leakage safeguard: select from the pre-decision state, execute the environment step, update from the censored observation, and only then retrieve the evaluator's full record. Each strategy receives its own environment and random-number generator.

In [ ]:
def run_strategy(strategy):
    environment = InventoryEnvironment(
        demand=demand_data["demand"],
        price=PRICE,
        unit_cost=UNIT_COST,
        holding_cost=HOLDING_COST,
        initial_inventory=INITIAL_INVENTORY,
    )
    state = environment.get_state()
    records = []
    cumulative_profit = 0.0
    while not environment.done:
        assert "demand" not in state
        action = strategy.select_action(state)
        observation = environment.step(action)
        assert "demand" not in observation
        strategy.update(observation)
        cumulative_profit += observation["profit"]
        record = environment.get_last_record()
        record["cumulative_profit"] = cumulative_profit
        records.append(record)
        state = environment.get_state()
    columns = [
        "day", "inventory_before", "order_up_to", "order_quantity",
        "demand", "sales", "inventory_after", "profit", "cumulative_profit",
    ]
    return pd.DataFrame(records)[columns], environment.profit_upper_bound()


strategy_factories = {
    "Epsilon-Greedy": lambda seed: EpsilonGreedy(ACTION_SET, rng=np.random.default_rng(seed)),
    "UCB": lambda seed: UCB(ACTION_SET, rng=np.random.default_rng(seed)),
    "Gradient Bandit": lambda seed: GradientBandit(ACTION_SET, rng=np.random.default_rng(seed)),
    "EWF": lambda seed: ExponentiallyWeightedForecaster(ACTION_SET, share_alpha=0.0, rng=np.random.default_rng(seed)),
    "FSF": lambda seed: FixedShareForecaster(ACTION_SET, rng=np.random.default_rng(seed)),
}

daily_results = {}
summary_rows = []
for strategy_name, factory in strategy_factories.items():
    daily, upper_bound = run_strategy(factory(RANDOM_SEED))
    daily.insert(0, "strategy", strategy_name)
    daily_results[strategy_name] = daily
    total_profit = float(daily["profit"].sum())
    summary_rows.append({
        "strategy": strategy_name,
        "total_profit": total_profit,
        "average_daily_profit": total_profit / N_DAYS,
        "profit_upper_bound": upper_bound,
        "percent_of_upper_bound": 100.0 * total_profit / upper_bound,
    })

comparison = pd.DataFrame(summary_rows)
comparison["rank"] = comparison["total_profit"].rank(ascending=False, method="min").astype(int)
comparison = comparison.sort_values(["rank", "strategy"]).reset_index(drop=True)
all_daily_results = pd.concat(daily_results.values(), ignore_index=True)

## Final results

The first table ranks the strategies. The second table contains every daily order-up-to decision and its realized outcome for all five strategies.

In [ ]:
display(comparison.round({
    "total_profit": 2,
    "average_daily_profit": 2,
    "profit_upper_bound": 2,
    "percent_of_upper_bound": 2,
}))
display(all_daily_results)

In [ ]:
figure, axes = plt.subplots(3, 1, figsize=(12, 14), sharex=True)
for strategy_name, daily in daily_results.items():
    axes[0].plot(daily["day"], daily["order_up_to"], marker="o", markersize=3, label=strategy_name)
    axes[1].plot(daily["day"], daily["profit"], marker="o", markersize=3, label=strategy_name)
    axes[2].plot(daily["day"], daily["cumulative_profit"], linewidth=2, label=strategy_name)
axes[0].set_ylabel("Order-up-to level")
axes[0].set_title("Daily decisions")
axes[1].set_ylabel("Daily profit")
axes[1].set_title("Daily profit")
axes[2].set_xlabel("Day")
axes[2].set_ylabel("Cumulative profit")
axes[2].set_title("Cumulative profit comparison")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend(ncol=3)
figure.tight_layout()
plt.show()

## Integrity checks

These checks confirm that all five strategies completed all 31 days, selected only valid levels, produced finite results, shared the same theoretical upper bound, and never required demand in their policy-facing inputs.

In [ ]:
expected_strategies = {"Epsilon-Greedy", "UCB", "Gradient Bandit", "EWF", "FSF"}
assert set(strategy_factories) == expected_strategies
assert set(daily_results) == expected_strategies
assert all(len(daily) == N_DAYS for daily in daily_results.values())
assert all(daily["order_up_to"].isin(ACTION_SET).all() for daily in daily_results.values())
assert np.isfinite(comparison.select_dtypes(include="number").to_numpy()).all()
assert comparison["profit_upper_bound"].nunique() == 1
print("All five strategies completed the leakage-safe 31-day evaluation.")